In [1]:
import numpy as np
import pandas as pd

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

print(df.isnull().sum())    # ดูว่ามี missing ตรงไหน

numerical = list(df.drop(columns='converted').select_dtypes(include='number').columns)
categorical = [c for c in df.columns if c not in numerical + ['converted']]
print('categorical:', categorical)
print('numerical  :', numerical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical  : ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


Q1 + Q2

In [2]:
print(df['industry'].value_counts().head())    # Q1

df[numerical].corr().round(3)                  # Q2

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


Split + seperate y

In [5]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values

df_train = df_train.drop(columns='converted')   # get target out from df
df_val = df_val.drop(columns='converted')

Q3 Mutual information

In [6]:
from sklearn.metrics import mutual_info_score

for col in categorical:
    print(col, round(mutual_info_score(y_train, df_train[col]), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


function train + measure accuracy

In [7]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

all_features = categorical + numerical

Q4 Accuracy

In [8]:
base_acc = train_and_score(all_features)
print(base_acc, f'{base_acc:.2f}')

0.645 0.65


Q5 Feature elimination

In [9]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    features = [x for x in all_features if x != f]
    acc = train_and_score(features)
    print(f, base_acc - acc)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


Q6 (select smallest value)

In [10]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(all_features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
